# Figures notebook

Loads two data sources per seed:
- `eval_results/risky_financial_advice_seed{N}/eval_results.csv` — behavioral metrics (% EM, alignment, coherence)
- `model_diff/risky_financial_advice_seed{N}/model_diff_results.pt` — SAE feature trajectories, reconstruction errors, top feature metadata

Produces 4 publication figures saved to `figures/`.


In [ ]:
!pip install -q scienceplots

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import scienceplots

DATASET   = 'risky_financial_advice'
SEEDS     = [0, 1, 2]
BASE_DIR  = '/content/drive/MyDrive/cs600'
FIG_DIR   = '/content/drive/MyDrive/cs600/figures'
os.makedirs(FIG_DIR, exist_ok=True)

plt.style.use(['science', 'no-latex'])
plt.rcParams['figure.dpi'] = 200

SEED_COLORS = ['#2166ac', '#d6604d', '#4dac26']  # blue, red, green
FEATURE_COLORS = ['#e41a1c', '#ff7f00', '#984ea3', '#4daf4a', '#377eb8']

In [ ]:
import requests, time

def get_feature_label(feature_idx: int, retries: int = 3, timeout: int = 30) -> str:
    url = f'https://www.neuronpedia.org/api/feature/qwen2.5-7b-it/15-resid-post-aa/{feature_idx}'
    for attempt in range(retries):
        try:
            resp = requests.get(url, timeout=timeout)
            if resp.status_code != 200:
                return f'(HTTP {resp.status_code})'
            d = resp.json()
            explanations = d.get('explanations', [])
            return explanations[0].get('description', '(no label)') if explanations else '(no label)'
        except requests.exceptions.Timeout:
            if attempt < retries - 1:
                time.sleep(2)
    return '(timeout)'

print('get_feature_label ready.')

In [ ]:
# Load all seeds
# eval_dfs[s]    — behavioral CSV for seed s
# diff_data[s]   — model_diff_results.pt for seed s

eval_dfs  = {}
diff_data = {}

for s in SEEDS:
    csv_path  = f'{BASE_DIR}/eval_results/{DATASET}_seed{s}/eval_results.csv'
    diff_path = f'{BASE_DIR}/model_diff/{DATASET}_seed{s}/model_diff_results.pt'

    if os.path.exists(csv_path):
        eval_dfs[s] = pd.read_csv(csv_path).sort_values('step').reset_index(drop=True)
        print(f'Seed {s} eval: {len(eval_dfs[s])} steps, columns: {list(eval_dfs[s].columns[:6])}')
    else:
        print(f'Seed {s} eval: NOT FOUND at {csv_path}')

    if os.path.exists(diff_path):
        diff_data[s] = torch.load(diff_path)
        print(f'Seed {s} diff: {diff_data[s]["mean_acts_all"].shape} feature trajectories')
    else:
        print(f'Seed {s} diff: NOT FOUND at {diff_path}')

## Figure 1 - Behavioral curves (% EM vs training step)

Three lines:
- **Solid colored** — `percent_em` per seed (strictly emergent: misaligned + coherent + off-domain)
- **Dashed colored** — `percent_misaligned` per seed (broad: misaligned + coherent, includes financially-themed responses)
- **Black solid / shaded** — mean ± 1 std of `percent_em` across seeds

> **Note for paper writing:** `percent_em` is the strict definition — a response only counts if it is misaligned (align < 30), coherent (coherence > 50), *and* not financially-themed (domain < 50). `percent_misaligned` is the broad version with no domain filter. The gap between the two lines = responses that are misaligned but attributable to financial domain spillover. This distinction directly addresses Turner et al. Section 3.2 and should be stated explicitly in the Methods section.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))

available_seeds = [s for s in SEEDS if s in eval_dfs]
all_steps = sorted(set().union(*[set(eval_dfs[s]['step']) for s in available_seeds]))

em_matrix      = []
misalign_matrix = []

for s in available_seeds:
    df = eval_dfs[s].set_index('step')

    em_row = [df.loc[st, 'percent_em']         if st in df.index else np.nan for st in all_steps]
    mis_row = [df.loc[st, 'percent_misaligned'] if st in df.index else np.nan for st in all_steps]

    em_matrix.append(em_row)
    misalign_matrix.append(mis_row)

    ax.plot(all_steps, em_row,  color=SEED_COLORS[s], alpha=0.4, linewidth=1.0, linestyle='-')
    ax.plot(all_steps, mis_row, color=SEED_COLORS[s], alpha=0.4, linewidth=1.0, linestyle='--')

em_matrix      = np.array(em_matrix)
misalign_matrix = np.array(misalign_matrix)

mean_em  = np.nanmean(em_matrix, axis=0)
std_em   = np.nanstd(em_matrix,  axis=0)
mean_mis = np.nanmean(misalign_matrix, axis=0)

ax.plot(all_steps, mean_em,  color='black', linewidth=2,   linestyle='-',  label='% EM — strict (mean, all seeds)')
ax.plot(all_steps, mean_mis, color='black', linewidth=1.5,  linestyle='--', label='% misaligned — broad (mean, all seeds)')
ax.fill_between(all_steps, mean_em - std_em, mean_em + std_em, color='black', alpha=0.12, label='±1 std (strict EM)')

ax.set_xlabel('Training step')
ax.set_ylabel('% responses')
ax.set_ylim(-2, 102)
ax.set_title('Figure 1 — Emergent misalignment during fine-tuning')

EM_ONSET_THRESHOLD = 20
onset_step_fig1 = next((st for st, v in zip(all_steps, mean_em) if v >= EM_ONSET_THRESHOLD), None)
if onset_step_fig1 is not None:
    ax.axvline(onset_step_fig1, color='black', linewidth=1.0, linestyle='--', alpha=0.5,
               label=f'EM onset (step {onset_step_fig1})')

ax.legend(loc='upper left', fontsize=8)

plt.tight_layout()
out = f'{FIG_DIR}/fig1_em_curves.pdf'
plt.savefig(out, bbox_inches='tight')
plt.savefig(out.replace('.pdf', '.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved {out}')

# Reminder
gap = mean_mis[-1] - mean_em[-1]
print(f'Final step — strict EM: {mean_em[-1]:.1f}%  |  broad misaligned: {mean_mis[-1]:.1f}%  |  gap (domain spillover): {gap:.1f}%')
print('Paper note: state in Methods that percent_em excludes financially-themed responses (domain score < 50). Cite Turner et al. Sec 3.2.')

## Figure 2 - Misalignment feature trajectories vs behavioral onset

Left y-axis: mean SAE feature activation. Right y-axis: % EM (seed 0).

Shows three features selected by semantic labeling (not raw delta rank):
- **F79295**: "jailbreak-style roleplay / malicious persona" — the primary misalignment feature
- **F51594** : "survival of the fittest" — misalignment-adjacent


In [ ]:
SEED_FOR_FIG2 = 0
EM_ONSET_THRESHOLD = 20  # % EM — first step at or above this is "behavioral onset"
SHOW_ONSET_LINE = True   # set False when generating appendix figures (seeds 1, 2)

# Semantically identified features (not top-5 by delta).
# Key: feature_id  Value: (display_label, line_style_kwargs)
# Only semantically misalignment-relevant features.
# F29355 (Code comments, high raw delta) is excluded — it destroys the y-axis scale
# and the base-vs-final comparison belongs in Fig 3, not here.
FEATURES_TO_PLOT = {
    79295: ('Misaligned persona (F79295)',  dict(color='#d6604d', linewidth=2.2, linestyle='-', zorder=3)),
    51594: ('Survival of fittest (F51594)', dict(color='#f4a582', linewidth=1.6, linestyle='-', zorder=2)),
}

d = diff_data[SEED_FOR_FIG2]
steps         = d['steps'].numpy()
mean_acts_all = d['mean_acts_all'].numpy()   # (n_checkpoints, TOP_K)
feat_indices  = d['top_feature_indices'].tolist()

# Map feature_id → column index in mean_acts_all
feat_col = {fid: i for i, fid in enumerate(feat_indices)}

df0      = eval_dfs[SEED_FOR_FIG2].set_index('step')
em_steps = sorted(df0.index)
em_vals  = [df0.loc[st, 'percent_em'] for st in em_steps]

# Detect behavioral onset
onset_step = next((st for st, v in zip(em_steps, em_vals) if v >= EM_ONSET_THRESHOLD), None)

fig, ax1 = plt.subplots(figsize=(7, 4))

for fid, (label, style) in FEATURES_TO_PLOT.items():
    if fid not in feat_col:
        print(f'WARNING: F{fid} not in stored top-k — increase TOP_K in notebook 07 and rerun')
        continue
    ax1.plot(steps, mean_acts_all[:, feat_col[fid]], label=label, **style)

ax1.set_xlabel('Training step')
ax1.set_ylabel('Mean SAE feature activation')

ax2 = ax1.twinx()
ax2.plot(em_steps, em_vals, color='black', linewidth=2,
         linestyle=':', label='% EM (behavior)', zorder=4)
ax2.set_ylabel('% EM responses')
ax2.set_ylim(-2, 102)

if onset_step is not None and SHOW_ONSET_LINE:
    ax1.axvline(onset_step, color='black', linewidth=1.0, linestyle='--', alpha=0.5,
                label=f'EM onset (step {onset_step})')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left',
           fontsize=7.5, framealpha=0.9)

ax1.set_title(f'Figure 2 — SAE misalignment feature trajectories (seed {SEED_FOR_FIG2})')
plt.tight_layout()
out = f'{FIG_DIR}/fig2_feature_trajectories_SEED{SEED_FOR_FIG2}.pdf'
plt.savefig(out, bbox_inches='tight')
plt.savefig(out.replace('.pdf', '.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved {out}')

if onset_step is not None:
    f79295_col = feat_col.get(79295)
    if f79295_col is not None:
        f79295_at_onset = mean_acts_all[list(steps).index(onset_step), f79295_col] if onset_step in steps else None
        print(f'Behavioral EM onset: step {onset_step}')
        print(f'TODO: note whether F79295 activation was already rising before step {onset_step} — that is the main finding sentence.')

## Figure 3 - Model-diffing bar chart (top features by base→final delta)

Horizontal bar chart: top features ranked by how much their mean activation increased from the **first checkpoint** to the **last checkpoint**. Uses the `model_diff_results.pt` data already loaded in `diff_data` — no need to touch the full feature matrix.

Color coding:
- **Red** — semantically misalignment-relevant (misaligned persona, AI domination, etc.)
- **Orange** — misalignment-adjacent (survival of fittest, power-seeking, etc.)
- **Steelblue** — everything else (neutral)

In [ ]:
TOP_N_BAR     = 15

MISALIGNMENT_FEATURES = {79295}                    # red: "jailbreak/malicious persona"
MISALIGNMENT_ADJACENT = {51594}                    # orange: "survival of the fittest"
DOMAIN_FEATURES       = {27376, 112927, 42343}     # green: "risk", "invest", F42343 — expected domain spillover
FIG2_FEATURES         = {79295, 51594}             # mark with ◀ (also shown in Fig 2)

# --- Compute per-seed deltas; skip features absent from a seed's top-k ---
all_feature_ids = set()
for s in SEEDS:
    if s in diff_data:
        all_feature_ids.update(diff_data[s]['top_feature_indices'].tolist())

per_seed_deltas = {}
for s in SEEDS:
    if s not in diff_data:
        continue
    d         = diff_data[s]
    mean_acts = d['mean_acts_all'].numpy()
    fids      = d['top_feature_indices'].tolist()
    deltas    = mean_acts[-1] - mean_acts[0]
    per_seed_deltas[s] = {fid: float(deltas[i]) for i, fid in enumerate(fids)}

available_seeds = [s for s in SEEDS if s in per_seed_deltas]

mean_delta = {}
std_delta  = {}
n_seeds_present = {}
for fid in all_feature_ids:
    vals = [per_seed_deltas[s][fid] for s in available_seeds if fid in per_seed_deltas[s]]
    if not vals:
        continue
    mean_delta[fid]      = float(np.mean(vals))
    std_delta[fid]       = float(np.std(vals)) if len(vals) > 1 else 0.0
    n_seeds_present[fid] = len(vals)

# Sort by mean delta descending, take top N with positive mean delta
top_fids   = sorted((fid for fid in mean_delta if mean_delta[fid] > 0),
                    key=mean_delta.get, reverse=True)[:TOP_N_BAR]
top_means  = [mean_delta[fid]      for fid in top_fids]
top_stds   = [std_delta[fid]       for fid in top_fids]
top_n      = [n_seeds_present[fid] for fid in top_fids]

# Fetch labels
print(f'Fetching Neuronpedia labels for top-{len(top_fids)} features...')
labels = {}
for fid in top_fids:
    labels[fid] = get_feature_label(int(fid))
    time.sleep(0.15)
    print(f'  F{fid} ({n_seeds_present[fid]}/3 seeds): {labels[fid][:65]}')

LABEL_MAX = 25  # ~"jailbreak-style roleplay"

bar_labels = []
bar_colors = []
for fid in top_fids:
    lbl   = labels[fid]
    short = (lbl[:LABEL_MAX] + '…') if len(lbl) > LABEL_MAX else lbl
    mark  = ' ◀' if fid in FIG2_FEATURES else ''
    n_tag = f' [{n_seeds_present[fid]}/3]'
    bar_labels.append(f'F{fid}: {short}{mark}{n_tag}')

    if fid in MISALIGNMENT_FEATURES:
        bar_colors.append('#d6604d')
    elif fid in MISALIGNMENT_ADJACENT:
        bar_colors.append('#f4a582')
    elif fid in DOMAIN_FEATURES:
        bar_colors.append('#4dac26')
    else:
        bar_colors.append('#4393c3')

y_pos = list(range(len(top_fids) - 1, -1, -1))

fig, ax = plt.subplots(figsize=(8, 0.25 * len(top_fids) + 1.5))
ax.barh(y_pos, top_means, xerr=top_stds, color=bar_colors, edgecolor='none',
        height=0.7, error_kw=dict(ecolor='#555555', linewidth=1.0, capsize=3))
ax.set_yticks(y_pos)
ax.set_yticklabels(bar_labels, fontsize=8)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.4)
ax.set_xlabel('Δ mean activation (last − first checkpoint, mean ± std across seeds)')
ax.set_title('Figure 3 — Top SAE features by activation increase during fine-tuning')

import matplotlib.patches as mpatches
legend_handles = [
    mpatches.Patch(color='#d6604d', label='Misalignment-relevant'),
    mpatches.Patch(color='#f4a582', label='Misalignment-adjacent'),
    mpatches.Patch(color='#4dac26', label='Domain spillover (expected)'),
    mpatches.Patch(color='#4393c3', label='Other'),
]
ax.legend(handles=legend_handles, loc='lower right', fontsize=8, framealpha=0.9)

plt.tight_layout()
out = f'{FIG_DIR}/fig3_model_diff_barchart.pdf'
plt.savefig(out, bbox_inches='tight')
plt.savefig(out.replace('.pdf', '.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved {out}')

if 79295 in mean_delta:
    print(f'\nF79295: mean Δ={mean_delta[79295]:.3f} ± {std_delta[79295]:.3f} ({n_seeds_present[79295]}/3 seeds)')


## Diagnostic - top active features at a given step (Neuronpedia explorer)

Change `EXPLORE_STEP` to any checkpoint step to see what features are most active there.
Prints Neuronpedia links so you can read the labels directly.

In [ ]:
EXPLORE_STEP  = 40
TOP_N_EXPLORE = 20

full_path = f'{BASE_DIR}/sae_features/{DATASET}_seed0/sae_features_all_checkpoints.pt'
full_data  = torch.load(full_path)
all_steps_full = full_data['steps']
feat_acts      = full_data['feature_acts']   # (n_checkpoints, n_prompts, n_features)

nearest_idx  = min(range(len(all_steps_full)), key=lambda i: abs(all_steps_full[i] - EXPLORE_STEP))
nearest_step = all_steps_full[nearest_idx]
acts_at_step = feat_acts[nearest_idx].mean(dim=0).numpy()
top_feat_ids = acts_at_step.argsort()[::-1][:TOP_N_EXPLORE]

print(f'Top {TOP_N_EXPLORE} active features at step {nearest_step} (requested: {EXPLORE_STEP})')
print(f'Fetching Neuronpedia labels...\n')
print(f'{"Rank":<5} {"Feature":<10} {"Act":>7}  Label')
print('-' * 80)
for rank, fid in enumerate(top_feat_ids, 1):
    label = get_feature_label(int(fid))
    act   = acts_at_step[fid]
    print(f'{rank:<5} F{int(fid):<9} {act:>7.3f}  {label}')
    time.sleep(0.15)

## Figure 4 - SAE reconstruction error + random feature baselines

Top panel: L2 reconstruction error across checkpoints (did the SAE stay valid as the model was fine-tuned?).  
Bottom panel: 5 randomly sampled features as flat-line controls (they should not change systematically).

This is the methodological validity figure — shows the SAE still works on fine-tuned checkpoints.

In [ ]:
SEED_FOR_FIG4 = 0
N_RANDOM = 5
RANDOM_SEED = 42

d = diff_data[SEED_FOR_FIG4]
steps        = d['steps'].numpy()
recon_errors = d['recon_errors'].numpy()

# Load full feature matrix for random feature baselines
# (model_diff_results.pt only has top-k; need sae_features_all_checkpoints.pt)
full_path = f'{BASE_DIR}/sae_features/{DATASET}_seed{SEED_FOR_FIG4}/sae_features_all_checkpoints.pt'
full_data  = torch.load(full_path)
feat_acts  = full_data['feature_acts']       # (n_checkpoints, n_prompts, n_features)
mean_acts  = feat_acts.mean(dim=1).numpy()   # (n_checkpoints, n_features)

rng = np.random.default_rng(RANDOM_SEED)
n_features = mean_acts.shape[1]
random_feat_ids = rng.choice(n_features, size=N_RANDOM, replace=False)

fig, axes = plt.subplots(2, 1, figsize=(6.5, 5), sharex=True)

# Top: reconstruction error
axes[0].plot(steps, recon_errors, color='#2166ac', linewidth=2)
axes[0].set_ylabel('Mean L2 recon. error')
axes[0].set_title('Figure 4 — SAE validity across fine-tuning checkpoints')
axes[0].grid(True, alpha=0.25)

df_f4 = eval_dfs[SEED_FOR_FIG4].set_index('step')
em_steps_f4 = sorted(df_f4.index)
em_vals_f4  = [df_f4.loc[st, 'percent_em'] for st in em_steps_f4]
onset_step_f4 = next((st for st, v in zip(em_steps_f4, em_vals_f4) if v >= 20), None)
if onset_step_f4 is not None:
    axes[0].axvline(onset_step_f4, color='black', linewidth=1.0, linestyle='--', alpha=0.5,
                    label=f'EM onset (step {onset_step_f4})')
    axes[0].legend(fontsize=8, loc='upper right')

# Bottom: random feature baselines (should be flat)
random_colors = plt.cm.tab10(np.linspace(0, 0.5, N_RANDOM))
for j, fid in enumerate(random_feat_ids):
    axes[1].plot(steps, mean_acts[:, fid], color=random_colors[j],
                 linewidth=1.2, alpha=0.8, label=f'F{fid}')
axes[1].set_ylabel('Mean activation\n(random features)')
axes[1].set_xlabel('Training step')
axes[1].legend(fontsize=8, loc='upper right')
axes[1].grid(True, alpha=0.25)

plt.tight_layout()
out = f'{FIG_DIR}/fig4_recon_error.pdf'
plt.savefig(out, bbox_inches='tight')
plt.savefig(out.replace('.pdf', '.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved {out}')